In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

df = pd.read_csv('../data/job_salary.csv')
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [2]:
df = df.dropna()
print(df.shape)

(250000, 10)


In [4]:
def get_seniority(title):
  title = str(title).lower()
  if any(x in title for x in ['junior','jr','entry']):
    return 1
  elif any(x in title for x in ['senior','sr','lead','principal']):
    return 3
  elif any(x in title for x in ['manager','director','head','vp']):
    return 4
  else:
    return 2

df['seniority'] = df['job_title'].apply(get_seniority)

In [7]:
print(df['education_level'].unique())
print(df['company_size'].unique())

['Bachelor' 'PhD' 'High School' 'Diploma' 'Master']
['Medium' 'Small' 'Large' 'Enterprise' 'Startup']


In [8]:
# Check what's actually in your data first, then set order
edu_order = [['High School', 'Diploma', 'Bachelor', 'Master', 'PhD']]
oe = OrdinalEncoder(categories=edu_order, handle_unknown='use_encoded_value', unknown_value=-1)
df['education_encoded'] = oe.fit_transform(df[['education_level']])

In [9]:
size_order = [['Small', 'Medium', 'Large']]
oe2 = OrdinalEncoder(categories=size_order, handle_unknown='use_encoded_value', unknown_value=-1)
df['company_size_encoded'] = oe2.fit_transform(df[['company_size']])

In [10]:
# Label encode — no order
le_industry = LabelEncoder()
df['industry_encoded'] = le_industry.fit_transform(df['industry'])

le_location = LabelEncoder()
df['location_encoded'] = le_location.fit_transform(df['location'])

le_title = LabelEncoder()
df['job_title_encoded'] = le_title.fit_transform(df['job_title'])

# Binary encode remote_work
df['remote_work'] = df['remote_work'].map({'Yes':1,'No':0})

In [11]:
df['salary_log'] = np.log1p(df['salary'])

features = ['experience_years','skills_count','certifications',
            'remote_work','seniority','education_encoded',
            'company_size_encoded','industry_encoded',
            'location_encoded','job_title_encoded']

X = df[features]
y = df['salary_log']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

# Save encoders for use in app later
joblib.dump(le_industry, '../model/le_industry.pkl')
joblib.dump(le_location, '../model/le_location.pkl')
joblib.dump(le_title, '../model/le_title.pkl')
joblib.dump(oe, '../model/oe_education.pkl')
joblib.dump(oe2, '../model/oe_company.pkl')

(200000, 10) (50000, 10)


['../model/oe_company.pkl']